In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Load dataset
df = pd.read_csv("Faceplate.csv")

# Ensure only color columns are used (binary values)
df = df[['Red', 'White', 'Blue', 'Green', 'Yellow']]

# Display first 10 transactions
print(df.head(10))

   Red  White  Blue  Green  Yellow
0    1      1     0      1       0
1    0      1     0      0       0
2    0      1     1      0       0
3    1      1     0      0       0
4    1      0     1      0       0
5    0      1     1      0       0
6    1      0     1      0       0
7    1      1     1      1       0
8    1      1     1      0       0
9    0      0     0      0       1


In [4]:
# Count transactions containing BOTH Red and White
both_count = ((df['Red'] == 1) & (df['White'] == 1)).sum()

# Total number of transactions
total_count = len(df)

# Support
support_red_white = both_count / total_count

print("Support of {Red, White} =", support_red_white)

Support of {Red, White} = 0.4


In [5]:
from mlxtend.frequent_patterns import apriori

# Generate frequent itemsets
frequent_itemsets = apriori(df, min_support=0.1, use_colnames=True)

print(frequent_itemsets)

    support                              itemsets
0       0.6                      frozenset({Red})
1       0.7                    frozenset({White})
2       0.6                     frozenset({Blue})
3       0.2                    frozenset({Green})
4       0.1                   frozenset({Yellow})
5       0.4               frozenset({Red, White})
6       0.4                frozenset({Red, Blue})
7       0.2               frozenset({Red, Green})
8       0.4              frozenset({White, Blue})
9       0.2             frozenset({White, Green})
10      0.1              frozenset({Green, Blue})
11      0.2         frozenset({Red, Blue, White})
12      0.2        frozenset({Red, Green, White})
13      0.1         frozenset({Red, Blue, Green})
14      0.1       frozenset({White, Blue, Green})
15      0.1  frozenset({Red, Blue, Green, White})


c:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [9]:
if 'Transaction' in df.columns:
    df = df.drop('Transaction', axis=1)

In [11]:
# Convert all columns to boolean
df_bool = df.astype(bool)


# Get frequent itemsets with minimum support 0.2
frequent_itemsets = apriori(df_bool, min_support=0.2, use_colnames=True)

# Display frequent itemsets and their support
print(frequent_itemsets)

    support                        itemsets
0       0.6                frozenset({Red})
1       0.7              frozenset({White})
2       0.6               frozenset({Blue})
3       0.2              frozenset({Green})
4       0.4         frozenset({Red, White})
5       0.4          frozenset({Red, Blue})
6       0.2         frozenset({Red, Green})
7       0.4        frozenset({White, Blue})
8       0.2       frozenset({White, Green})
9       0.2   frozenset({Red, Blue, White})
10      0.2  frozenset({Red, Green, White})


In [12]:

# Generate rules with minimum confidence 0.5
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

# Drop unneeded columns
rules = rules.drop(columns=['antecedent support', 'consequent support', 'conviction'])

# Sort by lift in descending order
rules = rules.sort_values(by='lift', ascending=False)

print(rules)

                  antecedents              consequents  support  confidence  \
14         frozenset({Green})  frozenset({Red, White})      0.2    1.000000   
12    frozenset({Red, White})       frozenset({Green})      0.2    0.500000   
4          frozenset({Green})         frozenset({Red})      0.2    1.000000   
13  frozenset({Green, White})         frozenset({Red})      0.2    1.000000   
7          frozenset({Green})       frozenset({White})      0.2    1.000000   
11    frozenset({Red, Green})       frozenset({White})      0.2    1.000000   
3           frozenset({Blue})         frozenset({Red})      0.4    0.666667   
2            frozenset({Red})        frozenset({Blue})      0.4    0.666667   
5          frozenset({White})        frozenset({Blue})      0.4    0.571429   
0            frozenset({Red})       frozenset({White})      0.4    0.666667   
1          frozenset({White})         frozenset({Red})      0.4    0.571429   
6           frozenset({Blue})       frozenset({White

In [13]:
top6_rules = rules.head(6)
print(top6_rules)

                  antecedents              consequents  support  confidence  \
14         frozenset({Green})  frozenset({Red, White})      0.2         1.0   
12    frozenset({Red, White})       frozenset({Green})      0.2         0.5   
4          frozenset({Green})         frozenset({Red})      0.2         1.0   
13  frozenset({Green, White})         frozenset({Red})      0.2         1.0   
7          frozenset({Green})       frozenset({White})      0.2         1.0   
11    frozenset({Red, Green})       frozenset({White})      0.2         1.0   

        lift  representativity  leverage  zhangs_metric   jaccard  certainty  \
14  2.500000               1.0      0.12          0.750  0.500000      1.000   
12  2.500000               1.0      0.12          1.000  0.500000      0.375   
4   1.666667               1.0      0.08          0.500  0.333333      1.000   
13  1.666667               1.0      0.08          0.500  0.333333      1.000   
7   1.428571               1.0      0.06      

In [14]:
# Take the top rule (highest lift)
top_rule = top6_rules.iloc[0]

antecedents = list(top_rule['antecedents'])
consequents = list(top_rule['consequents'])
confidence = top_rule['confidence'] * 100
lift = top_rule['lift']

sentence = f"If {antecedents} are purchased, then with confidence {confidence:.2f}% {consequents} will also be purchased. This rule has a lift ratio of {lift:.2f}."
print(sentence)

If ['Green'] are purchased, then with confidence 100.00% ['Red', 'White'] will also be purchased. This rule has a lift ratio of 2.50.


In [17]:
df = pd.read_csv("CharlesBookClub.csv")

# Columns to ignore (given in question)
cols_to_drop = ['Seq#', 'ID#', 'Gender', 'M', 'R', 'F',
                'FirstPurch', 'Related Purchase']

# Drop non-book columns
book_data = df.drop(columns=cols_to_drop, errors='ignore')

# Keep ONLY numeric columns (actual book purchase columns)
book_data = book_data.select_dtypes(include=['number'])

# Convert to binary incidence matrix
book_binary = (book_data > 0).astype(int)

book_binary.head(10)


,ChildBks,YouthBks,CookBks,DoItYBks,RefBks,ArtBks,GeogBks,ItalCook,ItalAtlas,ItalArt,Florence,Mcode,Rcode,Fcode,Yes_Florence,No_Florence
0,0,1,1,0,0,0,0,0,0,0,0,1,1,1,0,1
1,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1
2,1,1,1,0,1,0,1,1,0,0,0,1,1,1,0,1
3,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1
4,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1
5,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1
6,0,0,0,0,0,0,1,0,0,0,0,1,1,1,0,1
7,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1
8,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1
9,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,1


In [22]:
from mlxtend.frequent_patterns import apriori

# Ensure binary matrix (optional — already binary)
book_binary = (book_binary > 0)

# Convert 200 transactions → fraction
min_support = 200 / len(book_binary)

freq_books = apriori(book_binary,
                     min_support=min_support,
                     use_colnames=True)

print(f"Number of frequent itemsets found: {len(freq_books)}")

Number of frequent itemsets found: 935


In [25]:
from mlxtend.frequent_patterns import association_rules

# Generate rules with minimum confidence = 0.5
rules = association_rules(freq_books,
                          metric="confidence",
                          min_threshold=0.5)

# Keep only required columns
rules = rules[['antecedents', 'consequents','support','confidence','lift','leverage']]

# Sort by lift (highest first)
rules_sorted = rules.sort_values(by='lift', ascending=False)

# Display top 25 rules
top25_rules = rules_sorted.head(25)

print(top25_rules)

                                         antecedents  \
466                     frozenset({Fcode, Florence})   
468                        frozenset({Yes_Florence})   
469                            frozenset({Florence})   
463                        frozenset({Yes_Florence})   
462                     frozenset({Florence, Rcode})   
461                 frozenset({Yes_Florence, Rcode})   
464                            frozenset({Florence})   
4490         frozenset({Mcode, Fcode, Yes_Florence})   
4491             frozenset({Mcode, Fcode, Florence})   
4485  frozenset({Mcode, Fcode, Yes_Florence, Rcode})   
4492         frozenset({Mcode, Yes_Florence, Rcode})   
4486      frozenset({Mcode, Fcode, Florence, Rcode})   
4495         frozenset({Fcode, Yes_Florence, Rcode})   
4493             frozenset({Mcode, Florence, Rcode})   
4507                           frozenset({Florence})   
4496             frozenset({Fcode, Florence, Rcode})   
1888                frozenset({Fcode, Yes_Floren

c:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [27]:
from mlxtend.frequent_patterns import association_rules

rules_books = association_rules(freq_books,
                                metric="confidence",
                                min_threshold=0.5)

c:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [29]:
# Rule with highest support
rule_high_support = rules_books.sort_values('support', ascending=False).iloc[0]
print("Rule with highest support:")
print("Antecedents:", rule_high_support['antecedents'])
print("Consequents:", rule_high_support['consequents'])
print("Support:", rule_high_support['support'])
print("Confidence:", rule_high_support['confidence'])
print("Lift:", rule_high_support['lift'])

Rule with highest support:
Antecedents: frozenset({'Mcode'})
Consequents: frozenset({'Rcode'})
Support: 1.0
Confidence: 1.0
Lift: 1.0


In [30]:
# Rule with highest support
rule_high_support = rules_books.sort_values('support',
                                            ascending=False).iloc[0]

# Rule with highest lift
rule_high_lift = rules_books.sort_values('lift',
                                         ascending=False).iloc[0]

print("=== Rule with Highest Support ===")
print("Antecedents:", rule_high_support['antecedents'])
print("Consequents:", rule_high_support['consequents'])
print("Support:", rule_high_support['support'])
print("Confidence:", rule_high_support['confidence'])
print("Lift:", rule_high_support['lift'])

print("\n=== Rule with Highest Lift ===")
print("Antecedents:", rule_high_lift['antecedents'])
print("Consequents:", rule_high_lift['consequents'])
print("Support:", rule_high_lift['support'])
print("Confidence:", rule_high_lift['confidence'])
print("Lift:", rule_high_lift['lift'])

# Comparison of support values
print("\n=== Support Comparison ===")
print("Support (Highest-support rule):", rule_high_support['support'])
print("Support (Highest-lift rule):", rule_high_lift['support'])

# Trade-off explanation (printed)
print("\n=== Trade-off Discussion ===")
print("The rule with the highest lift is more efficient at predicting")
print("the consequent but usually applies to fewer transactions (low support).")
print("The rule with the highest support affects many transactions")
print("but may represent a weaker association (lower lift).")
print("Thus, there is a trade-off between rule strength (lift)")
print("and coverage (support).")

=== Rule with Highest Support ===
Antecedents: frozenset({'Mcode'})
Consequents: frozenset({'Rcode'})
Support: 1.0
Confidence: 1.0
Lift: 1.0

=== Rule with Highest Lift ===
Antecedents: frozenset({'Fcode', 'Florence'})
Consequents: frozenset({'Yes_Florence'})
Support: 0.0845
Confidence: 1.0
Lift: 11.834319526627219

=== Support Comparison ===
Support (Highest-support rule): 1.0
Support (Highest-lift rule): 0.0845

=== Trade-off Discussion ===
The rule with the highest lift is more efficient at predicting
the consequent but usually applies to fewer transactions (low support).
The rule with the highest support affects many transactions
but may represent a weaker association (lower lift).
Thus, there is a trade-off between rule strength (lift)
and coverage (support).


In [ ]:

top10_lift = rules_books.sort_values('lift', ascending=False).head(10)

print("=== Top 10 Rules by Lift ===")
for i, row in enumerate(top10_lift.itertuples(), 1):
    print(f"Rule {i}:")
    print("  Antecedents:", row.antecedents)
    print("  Consequents:", row.consequents)
    print("  Support:", row.support)
    print("  Confidence:", row.confidence)
    print("  Lift:", row.lift)
    print("  Leverage:", row.leverage)
    print("-"*40)


lowest_conf_rule = top10_lift.sort_values('confidence').iloc[0]

print("\n=== Rule with Lowest Confidence among Top 10 Lift Rules ===")
print("Antecedents:", lowest_conf_rule['antecedents'])
print("Consequents:", lowest_conf_rule['consequents'])
print("Support:", lowest_conf_rule['support'])
print("Confidence:", lowest_conf_rule['confidence'])
print("Lift:", lowest_conf_rule['lift'])
print("Leverage:", lowest_conf_rule['leverage'])

=== Top 10 Rules by Lift ===
Rule 1:
  Antecedents: frozenset({'Fcode', 'Florence'})
  Consequents: frozenset({'Yes_Florence'})
  Support: 0.0845
  Confidence: 1.0
  Lift: 11.834319526627219
  Leverage: 0.07735975
----------------------------------------
Rule 2:
  Antecedents: frozenset({'Yes_Florence'})
  Consequents: frozenset({'Fcode', 'Florence'})
  Support: 0.0845
  Confidence: 1.0
  Lift: 11.834319526627219
  Leverage: 0.07735975
----------------------------------------
Rule 3:
  Antecedents: frozenset({'Florence'})
  Consequents: frozenset({'Fcode', 'Yes_Florence'})
  Support: 0.0845
  Confidence: 1.0
  Lift: 11.834319526627219
  Leverage: 0.07735975
----------------------------------------
Rule 4:
  Antecedents: frozenset({'Yes_Florence'})
  Consequents: frozenset({'Florence', 'Rcode'})
  Support: 0.0845
  Confidence: 1.0
  Lift: 11.834319526627219
  Leverage: 0.07735975
----------------------------------------
Rule 5:
  Antecedents: frozenset({'Florence', 'Rcode'})
  Consequen

In [34]:


# Repeatable randomness
np.random.seed(0)

# Parameters
num_transactions = 50
num_items = 9

# Create random 0/1 binary matrix
data_random = np.random.randint(2, size=(num_transactions, num_items))

# Convert to DataFrame with item names Item1…Item9
items = [f"Item{i}" for i in range(1, num_items+1)]
df_random = pd.DataFrame(data_random, columns=items)

print("=== First 10 Transactions of Synthetic Dataset ===")
print(df_random.head(10))

=== First 10 Transactions of Synthetic Dataset ===
   Item1  Item2  Item3  Item4  Item5  Item6  Item7  Item8  Item9
0      0      1      1      0      1      1      1      1      1
1      1      1      0      0      1      0      0      0      0
2      0      1      0      1      1      0      0      1      1
3      1      1      0      1      0      1      0      1      1
4      0      1      1      0      0      1      0      1      1
5      1      1      1      0      1      0      1      1      1
6      1      0      1      0      0      1      1      0      1
7      0      1      0      0      0      0      0      1      1
8      0      0      0      1      1      0      1      0      0
9      1      0      1      1      1      1      1      1      0


In [35]:
# Minimum support = 2 transactions / 50 = 0.04
min_support = 2 / num_transactions

# Frequent itemsets
freq_items = apriori(df_random, min_support=min_support, use_colnames=True)

print("\nNumber of frequent itemsets found:", len(freq_items))

# Generate association rules, min confidence = 0.7
rules_random = association_rules(freq_items, metric="confidence", min_threshold=0.7)

print("\nTotal association rules generated:", len(rules_random))


Number of frequent itemsets found: 358

Total association rules generated: 377


c:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [36]:
# Sort by lift descending
top6_lift = rules_random.sort_values('lift', ascending=False).head(6)

# Display only required columns
top6_display = top6_lift[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

print("\n=== Top 6 Rules by Lift ===")
for i, row in enumerate(top6_display.itertuples(), 1):
    print(f"Rule {i}:")
    print("  Antecedents:", row.antecedents)
    print("  Consequents:", row.consequents)
    print("  Support:", row.support)
    print("  Confidence:", row.confidence)
    print("  Lift:", row.lift)
    print("-"*40)


=== Top 6 Rules by Lift ===
Rule 1:
  Antecedents: frozenset({'Item6', 'Item9', 'Item8', 'Item7'})
  Consequents: frozenset({'Item3', 'Item5', 'Item2'})
  Support: 0.04
  Confidence: 1.0
  Lift: 5.555555555555555
----------------------------------------
Rule 2:
  Antecedents: frozenset({'Item9', 'Item2', 'Item3', 'Item5', 'Item6'})
  Consequents: frozenset({'Item8', 'Item7'})
  Support: 0.04
  Confidence: 1.0
  Lift: 5.0
----------------------------------------
Rule 3:
  Antecedents: frozenset({'Item4', 'Item3', 'Item5', 'Item6', 'Item7'})
  Consequents: frozenset({'Item8', 'Item1'})
  Support: 0.04
  Confidence: 1.0
  Lift: 5.0
----------------------------------------
Rule 4:
  Antecedents: frozenset({'Item2', 'Item1', 'Item3', 'Item6', 'Item8'})
  Consequents: frozenset({'Item4', 'Item7'})
  Support: 0.04
  Confidence: 1.0
  Lift: 4.545454545454546
----------------------------------------
Rule 5:
  Antecedents: frozenset({'Item5', 'Item4', 'Item2', 'Item7'})
  Consequents: frozenset

In [37]:

np.random.seed(0)

# Parameters
num_users = 1000
num_items = 100
num_ratings = 5000

# Generate random synthetic ratings
data = {
    "userID": np.random.randint(0, num_users, size=num_ratings),
    "itemID": np.random.randint(0, num_items, size=num_ratings),
    "rating": np.random.randint(1, 6, size=num_ratings)  # ratings 1-5
}

df_ratings = pd.DataFrame(data)

print("=== First 10 rows of synthetic ratings dataset ===")
print(df_ratings.head(10))

=== First 10 rows of synthetic ratings dataset ===
   userID  itemID  rating
0     684      49       2
1     559      63       3
2     629       9       5
3     192      24       1
4     835      68       4
5     763      26       5
6     707      52       1
7     359      54       1
8       9      85       4
9     723      78       5
